In [4]:
import numpy as np
import pandas as pd
from flaml import AutoML
from sklearn.model_selection import train_test_split, cross_val_predict, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
from scipy.stats import spearmanr

warnings.filterwarnings("ignore")

DATA_PATH = "data/ZTF_DESI_matched_lc_cuts_z_cuts_hostprop_cleaned.csv"
df = pd.read_csv(DATA_PATH)

FEATURE_COLS = [
    "DESI_FASTSPEC_LOGMSTAR",
    "DESI_FASTSPEC_SFR",
    "DESI_FASTSPEC_VDISP",
    "DESI_FASTSPEC_DN4000",
    "DESI_FASTSPEC_g_minus_r",
    "DESI_FASTSPEC_AGE",
]

X = df[FEATURE_COLS]
y = df["residuals"]
w = 1.0 / df["ZTF_sigma_mu"] ** 2

X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    X, y, w, test_size=0.2, random_state=42
)

print(f"{len(df)} SNe Ia, {len(FEATURE_COLS)} features")

automl = AutoML()

automl.fit(
    X_train, y_train,
    sample_weight=w_train.values,
    task="regression",
    metric="rmse",
    time_budget=600,
    estimator_list=[
        "rf",          # Random Forest
        "xgboost",     # XGBoost
        "extra_tree",  # Extra Trees
        "lrl2",        # Ridge (linear baseline)
        "lrl1",        # Lasso (linear baseline)
    ],
    eval_method="cv",
    n_splits=5,
    seed=42,
    verbose=2,  # set to 0 if you don't want the play-by-play
)

print(f"\nBest model:   {automl.best_estimator}")
print(f"Best CV RMSE: {-automl.best_loss:.5f}")
print(f"Best config:  {automl.best_config}")


def nmad(residuals):
    return 1.4826 * np.median(np.abs(residuals - np.median(residuals)))

# FLAML stores the best model per estimator type
results = []
for est_name, est_info in automl.best_config_per_estimator.items():
    if est_info is None:
        continue
    model = automl.best_model_for_estimator(est_name)
    y_pred = model.predict(X_test)
    resid = y_test - y_pred
    results.append({
        "Model": est_name,
        "Test RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
        "Test MAE": mean_absolute_error(y_test, y_pred),
        "Test R²": r2_score(y_test, y_pred),
        "Test NMAD": nmad(resid),
    })

df_results = pd.DataFrame(results).sort_values("Test RMSE")
print(df_results.to_string(index=False, float_format="%.4f"))


521 SNe Ia, 6 features


KeyboardInterrupt: 

In [1]:
from pycaret.regression import *


import numpy as np
import pandas as pd
from flaml import AutoML
from sklearn.model_selection import train_test_split, cross_val_predict, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
from scipy.stats import spearmanr
warnings.filterwarnings("ignore")

DATA_PATH = "data/ZTF_DESI_matched_lc_cuts_z_cuts_hostprop_cleaned.csv"
df = pd.read_csv(DATA_PATH)

FEATURE_COLS = [
    "DESI_FASTSPEC_LOGMSTAR",
    "DESI_FASTSPEC_SFR",
    "DESI_FASTSPEC_VDISP",
    "DESI_FASTSPEC_DN4000",
    "DESI_FASTSPEC_g_minus_r",
    "DESI_FASTSPEC_AGE",
]

X = df[FEATURE_COLS]
y = df["residuals"]
w = 1.0 / df["ZTF_sigma_mu"] ** 2

X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    X, y, w, test_size=0.2, random_state=42
)

print(f"{len(df)} SNe Ia, {len(FEATURE_COLS)} features")

df_pycaret = df[FEATURE_COLS + ["residuals"]].copy()

setup(df_pycaret, target="residuals", session_id=42, fold=5, verbose=False)

best = compare_models(
    include=["ridge", "en", "rf", "xgboost", "et", "mlp"],
    sort="RMSE",
)

521 SNe Ia, 6 features


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
ridge,Ridge Regression,0.1352,0.0306,0.1741,0.0633,0.1252,1.9293,0.4000
en,Elastic Net,0.1419,0.0333,0.1816,-0.0192,0.1445,19.1218,0.2100
rf,Random Forest Regressor,0.1453,0.0339,0.1837,-0.0495,0.1226,133.5832,0.2160
et,Extra Trees Regressor,0.1502,0.0357,0.1886,-0.1055,0.1236,61.9715,0.2340
mlp,MLP Regressor,0.1516,0.0375,0.1925,-0.1459,0.1229,63.3660,0.0120
xgboost,Extreme Gradient Boosting,0.1617,0.0420,0.2042,-0.3092,0.1233,114.3461,0.2100


In [ ]:
from pycaret.regression import *

df_pycaret = df[FEATURE_COLS + ["residuals"]].copy()
setup(df_pycaret, target="residuals", session_id=42, fold=5, verbose=False)

# Compare with default params first
best = compare_models(
    include=["ridge", "en", "rf", "xgboost", "et", "mlp"],
    sort="RMSE",
    n_select=6,  # select ALL 6 so we can tune all of them
)

# Tune all 6
tuned = [tune_model(m, optimize="RMSE", n_iter=500, search_library="optuna") for m in best]


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
ridge,Ridge Regression,0.1352,0.0306,0.1741,0.0633,0.1252,1.9293,0.0060
en,Elastic Net,0.1419,0.0333,0.1816,-0.0192,0.1445,19.1218,0.0040
rf,Random Forest Regressor,0.1453,0.0339,0.1837,-0.0495,0.1226,133.5832,0.2260
et,Extra Trees Regressor,0.1502,0.0357,0.1886,-0.1055,0.1236,61.9715,0.0160
mlp,MLP Regressor,0.1516,0.0375,0.1925,-0.1459,0.1229,63.3660,0.2020
xgboost,Extreme Gradient Boosting,0.1617,0.0420,0.2042,-0.3092,0.1233,114.3461,0.0120


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.1477,0.0391,0.1978,0.0901,0.1464,1.0200
1,0.1203,0.0233,0.1525,0.0897,0.1053,22.0986
2,0.1279,0.0277,0.1665,0.0194,0.1191,2.0929
3,0.1518,0.0360,0.1898,0.0399,0.1425,1.2133
4,0.1274,0.0266,0.1631,0.0894,0.1181,2.6934
Mean,0.1350,0.0306,0.1740,0.0657,0.1263,5.8236
Std,0.0124,0.0060,0.0171,0.0301,0.0157,8.1599


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.1477,0.0391,0.1978,0.0900,0.1465,1.0180
1,0.1202,0.0233,0.1525,0.0896,0.1053,21.8661
2,0.1279,0.0277,0.1665,0.0187,0.1191,2.0981
3,0.1519,0.0361,0.1900,0.0385,0.1426,1.2107
4,0.1274,0.0266,0.1631,0.0896,0.1182,2.6636
Mean,0.1350,0.0306,0.1740,0.0653,0.1264,5.7713
Std,0.0124,0.0060,0.0171,0.0306,0.0157,8.0695


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.1449,0.0383,0.1958,0.1087,0.1467,0.9851
1,0.1237,0.0243,0.1560,0.0477,0.1070,178.2178
2,0.1307,0.0286,0.1690,-0.0107,0.1177,2.4989
3,0.1518,0.0345,0.1856,0.0819,0.1398,1.2437
4,0.1304,0.0276,0.1662,0.0537,0.1160,3.3897
Mean,0.1363,0.0307,0.1745,0.0563,0.1254,37.2670
Std,0.0104,0.0050,0.0143,0.0399,0.0152,70.4807


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.1478,0.0389,0.1973,0.0951,0.1485,1.0293
1,0.1221,0.0239,0.1547,0.0638,0.1103,74.1189
2,0.1288,0.0280,0.1674,0.0087,0.1221,2.1028
3,0.1499,0.0340,0.1844,0.0942,0.1421,1.2657
4,0.1292,0.0263,0.1623,0.0977,0.1235,3.8047
Mean,0.1356,0.0302,0.1732,0.0719,0.1293,16.4643
Std,0.0112,0.0055,0.0155,0.0339,0.0140,28.8437


,,
,,
Initiated,. . . . . . . . . . . . . . . . . .,12:03:16
Status,. . . . . . . . . . . . . . . . . .,Searching Hyperparameters
Estimator,. . . . . . . . . . . . . . . . . .,MLP Regressor


Processing:   0%|          | 0/7 [00:00<?, ?it/s]

# Spearman Rho #

Four of the six DESI host galaxy properties show statistically significant monotonic correlations with the Hubble residuals: g−r (ρ = −0.303), DN4000 (−0.276), log M★ (−0.269), and Age (−0.219), all with p < 10⁻⁷. The negative sign means redder, older, more massive hosts with stronger 4000 Å breaks produce SNe Ia that are brighter after Tripp correction — consistent with the direction of the known mass step. Notably, g−r is the strongest correlate, not stellar mass, echoing Kelsey et al. (2023)'s finding that galaxy color removes more Hubble residual dispersion than mass. Velocity dispersion and SFR show no significant individual correlation (p > 0.29), though they may still carry information through interactions with other features.

In [2]:


# ──────────────────────────────────────────────────────────────
# Spearman rho: HOST GALAXY properties vs. Hubble residuals
# ──────────────────────────────────────────────────────────────
spearman_rho_logmstar, p_logmstar = spearmanr(df["DESI_FASTSPEC_LOGMSTAR"], df["residuals"])
print("Spearman ρ for LOGMSTAR vs. Δμ:", spearman_rho_logmstar, "p-value:", p_logmstar)

spearman_rho_sfr, p_sfr = spearmanr(df["DESI_FASTSPEC_SFR"], df["residuals"])
print("Spearman ρ for SFR vs. Δμ:", spearman_rho_sfr, "p-value:", p_sfr)

spearman_rho_vdisp, p_vdisp = spearmanr(df["DESI_FASTSPEC_VDISP"], df["residuals"])
print("Spearman ρ for VDISP vs. Δμ:", spearman_rho_vdisp, "p-value:", p_vdisp)

spearman_rho_dn4000, p_dn4000 = spearmanr(df["DESI_FASTSPEC_DN4000"], df["residuals"])
print("Spearman ρ for DN4000 vs. Δμ:", spearman_rho_dn4000, "p-value:", p_dn4000)

spearman_rho_gr, p_gr = spearmanr(df["DESI_FASTSPEC_g_minus_r"], df["residuals"])
print("Spearman ρ for g-r vs. Δμ:", spearman_rho_gr, "p-value:", p_gr)

spearman_rho_age, p_age = spearmanr(df["DESI_FASTSPEC_AGE"], df["residuals"])
print("Spearman ρ for AGE vs. Δμ:", spearman_rho_age, "p-value:", p_age)

Spearman ρ for LOGMSTAR vs. Δμ: -0.2692462125647644 p-value: 4.194362399287488e-10
Spearman ρ for SFR vs. Δμ: 0.015072438818078971 p-value: 0.7314269576781596
Spearman ρ for VDISP vs. Δμ: -0.04626396665481723 p-value: 0.2918725084635683
Spearman ρ for DN4000 vs. Δμ: -0.27643720587661286 p-value: 1.3612568044559e-10
Spearman ρ for g-r vs. Δμ: -0.30319965515544306 p-value: 1.5363951329768912e-12
Spearman ρ for AGE vs. Δμ: -0.21910442704179606 p-value: 4.403399529561777e-07
